## Cleaning

### Data Import

In [94]:
import polars as pl
import altair as alt
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # override=True ensures .env values take precedence over system env vars

USER = os.getenv("USER")
PASSWORD = os.getenv("PW")
HOST = os.getenv("HOST")
DATABASE = os.getenv("DB_NAME")

uri = f"postgresql://{USER}:{PASSWORD}@{HOST}:5432/{DATABASE}"

query = "SELECT * FROM daan_822.article_detailed"

df = pl.read_database_uri(query=query, uri=uri)

In [95]:
df.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,license_type,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""0""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""7178""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,null,null,53.387852,null,null,null,null,null,null,null,0.657147,null,null,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""",""" Helicobacter pylori Screeni…","""""","""[]""","""2024-10-08""","""["" Artificial intelligence"",""S…",0.0,null,"""AACE Endocrinology and Diabete…","""""","""""","""""","""""","""[""*Brain Trauma Foundation, Pa…",0.0,"""0""","""0""","""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,null,null,31.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,43.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,59.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-5-12""","""[]""",1194.0,null,"""npj Metabolic Health and Disea…","""ecancer Global Foundation""","""©Zongqi Xia, Prerna Chikersal,…","""2026""","""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,"""9""","""9""","""[]""","""[]""",1.0,"""[]""",1.0


### Cleaning
1. Whitespace in strings — strips leading/trailing whitespace from all string columns
  2. Empty strings instead of nulls — replaces "" with None for proper null semantics
  3. Wrong types for numeric columns — copyright_year, figure_count, table_count stored as Text in DB, cast to Int16
  4. pub_date stored as string — converts to proper Date type with multi-format fallback (YYYY-MM-DD, YYYY-MM, YYYY)
  5. license_type entirely null — drops the column
  6. Case inconsistency — lowercases keywords, journal_title, publisher_name
  7. JSON-as-string fields — parses authors and keywords from JSON strings into structured columns
  8. Remaining JSON array columns — detects them dynamically and converts to comma-separated strings

In [96]:
# cleaning whitespace and replacing blank with true nulls
df_cleaning = df.with_columns(pl.selectors.string().str.strip_chars().replace("", None))

# converting string type to int
df_cleaning = df_cleaning.with_columns(
    pl.selectors.by_name("copyright_year", "figure_count", "table_count").cast(pl.Int16)
)

# converting pub_date to date
df_cleaning = df_cleaning.with_columns(
    pl.coalesce(
        [
            pl.col("pub_date").str.to_date("%Y-%m-%d", strict=False),
            pl.col("pub_date").str.to_date("%Y-%m", strict=False),
            pl.col("pub_date").str.to_date("%Y", strict=False),
        ]
    )
)

# dropping column as it is null
df_cleaning = df_cleaning.drop("license_type")

# making columns lower case
df_cleaning = df_cleaning.with_columns(pl.selectors.by_name("keywords", "journal_title", "publisher_name").str.to_lowercase())

In [97]:
df_cleaning.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,f64,str,str,f64,f64,f64,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7140""","""7178""","""7156""","""7178""",7178.0,"""7178""","""6518""","""6871""",6105.0,"""7117""","""7178""",7178.0,7178.0,7178.0,"""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""38""","""0""","""22""","""0""",0.0,"""0""","""660""","""307""",1073.0,"""61""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,"""2025-07-02 23:04:15.561766""",null,53.387852,null,null,null,2024.984767,null,null,0.657147,3.942045,3.011702,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,0.150133,null,null,null,7.704139,2.888119,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""","""10 Practical Considerations fo…","""10""","""[]""","""2024-03-20""","""["" artificial intelligence"",""s…",0.0,"""aace endocrinology and diabete…","""a k peters/crc press""","""(c) 2025 The authors; licensee…",2023.0,"""(1) Background: Alveolar bone …","""[""*Brain Trauma Foundation, Pa…",0.0,0.0,0.0,"""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,"""2025-04-04""",null,31.0,null,null,null,2025.0,null,null,null,2.0,2.0,null,null,null,null,null
"""50%""",null,null,null,null,null,null,"""2025-07-07""",null,43.0,null,null,null,2025.0,null,null,null,3.0,3.0,null,null,null,null,null
"""75%""",null,null,null,null,null,null,"""2025-10-06""",null,59.0,null,null,null,2025.0,null,null,null,5.0,4.0,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-05-12""","""[]""",1194.0,"""yonsei medical journal""","""zenodo""","""©Zongqi Xia, Prerna Chikersal,…",2026.0,"""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,487.0,77.0,"""[]""","""[]""",1.0,"""[]""",1.0


In [98]:
# labelling field names in the json construct for authors
dtypes = pl.List(
    pl.Struct(
        [
            pl.Field("given_names", pl.String),
            pl.Field("is_corresponding", pl.Boolean),
            pl.Field("orcid", pl.String),
            pl.Field("surname", pl.String),
        ]
    )
)

# parse JSON columns without exploding to avoid author x keyword cartesian product
df_parsed = df_cleaning.with_columns(
    pl.col("authors").str.json_decode(dtypes).alias("authors_struct"),
    pl.col("keywords").str.json_decode(pl.List(pl.String)).alias("keywords_list"),
).drop("authors", "keywords")

# identify columns that are json formatted in order to clean to be comma separated
json_array_cols = [
    col
    for col, dtype in df_parsed.schema.items()
    if dtype == pl.String
    and df_parsed.select(pl.col(col).drop_nulls().str.starts_with("[").all()).item()
]

# clean up JSON array columns to comma-separated strings
df_base = df_parsed.with_columns(
    [
        pl.col(col).str.json_decode(pl.List(pl.String)).list.join(", ").alias(col)
        for col in json_array_cols
    ]
)

# separate exploded views to avoid author x keyword cartesian product
df_authors = (
    df_base.explode("authors_struct")
    .unnest("authors_struct")
    .with_columns(
        (pl.col("given_names") + " " + pl.col("surname")).alias("author_full_name")
    )
)

df_keywords = df_base.explode("keywords_list").rename({"keywords_list": "keyword"})

In [99]:
df_base.describe()

statistic,pmid,doi,article_type,article_title,article_subject,pub_date,reference_count,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available,authors_struct,keywords_list
str,str,str,str,str,str,str,f64,str,str,str,f64,str,str,f64,f64,f64,str,str,f64,str,f64,f64,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7140""","""7156""",7178.0,"""7178""","""6518""","""6871""",6105.0,"""7117""","""7178""",7178.0,7178.0,7178.0,"""7178""","""7178""",7178.0,"""7178""",7178.0,7178.0,7178.0
"""null_count""","""0""","""11""","""0""","""0""","""38""","""22""",0.0,"""0""","""660""","""307""",1073.0,"""61""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,"""0""",0.0,0.0,0.0
"""mean""",null,null,null,null,null,"""2025-07-02 23:04:15.561766""",53.387852,null,null,null,2024.984767,null,null,0.657147,3.942045,3.011702,null,null,0.395653,null,0.033853,null,null
"""std""",null,null,null,null,null,null,48.141303,null,null,null,0.150133,null,null,null,7.704139,2.888119,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""","""10 Practical Considerations fo…","""10""","""2024-03-20""",0.0,"""aace endocrinology and diabete…","""a k peters/crc press""","""(c) 2025 The authors; licensee…",2023.0,"""(1) Background: Alveolar bone …","""""",0.0,0.0,0.0,"""""","""""",0.0,"""""",0.0,null,null
"""25%""",null,null,null,null,null,"""2025-04-04""",31.0,null,null,null,2025.0,null,null,null,2.0,2.0,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,"""2025-07-07""",43.0,null,null,null,2025.0,null,null,null,3.0,3.0,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,"""2025-10-06""",59.0,null,null,null,2025.0,null,null,null,5.0,4.0,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""2026-05-12""",1194.0,"""yonsei medical journal""","""zenodo""","""©Zongqi Xia, Prerna Chikersal,…",2026.0,"""​​BackgroundOsteoporosis, a pr…","""‡Department of Neurosurgery, V…",1.0,487.0,77.0,"""贵州省第八批高层次创新型“百”层次人才""","""“For ethical and administrativ…",1.0,"""eHarmonize is an open-source p…",1.0,null,null


In [100]:
cols_with_null_info = [
    (col, df_base.select(pl.col(f"{col}")).null_count().item(), df_base.schema[col])
    for col in df_base.columns
    if df_base.select(pl.col(col).null_count()).item() > 0
]

cols_with_null_info

[('doi', 11, String),
 ('article_subject', 38, String),
 ('pub_date', 22, Date),
 ('publisher_name', 660, String),
 ('copyright_statement', 307, String),
 ('copyright_year', 1073, Int16),
 ('abstract', 61, String)]

#### Copyright Year validation

In [ ]:
# extracts year from the copyright statement
df_base = df_base.with_columns(
    pl.col("copyright_statement")
    .str.extract(r"((?:19|20)\d{2})")
    .cast(pl.Int16, strict=False)
    .alias("year_from_statement")
)

# compares the result
df_base = df_base.with_columns(
    (pl.col("copyright_year") == pl.col("year_from_statement")).alias("copyright_year_match")
)

# if the year extracted is not null, it replaces the copyright year
df_base = df_base.with_columns(
    pl.when(pl.col("year_from_statement").is_not_null())
    .then(pl.col("year_from_statement"))
    .otherwise(pl.col("copyright_year"))
    .alias("copyright_year")
)

# there are only 25 rows that did not match when compared
copyright_year_mismatch_df = df_base.filter(pl.col("copyright_year_match") == False)

df_base = df_base.with_columns(
    pl.col("pub_date").dt.year().cast(pl.Int16).alias("pub_year")
).with_columns(
    (pl.col("copyright_year") == pl.col("pub_year")).alias("copyright_eq_pub")
)

## Analysis

In [68]:
df_base.schema

Schema([('pmid', String),
        ('doi', String),
        ('article_type', String),
        ('article_title', String),
        ('article_subject', String),
        ('pub_date', Date),
        ('reference_count', Int32),
        ('journal_title', String),
        ('publisher_name', String),
        ('copyright_statement', String),
        ('copyright_year', Int16),
        ('abstract', String),
        ('affiliations', String),
        ('has_supplemental', Boolean),
        ('figure_count', Int16),
        ('table_count', Int16),
        ('funding', String),
        ('data_available_details', String),
        ('data_available', Boolean),
        ('code_available_details', String),
        ('code_available', Boolean),
        ('authors_struct',
         List(Struct({'given_names': String, 'is_corresponding': Boolean, 'orcid': String, 'surname': String}))),
        ('keywords_list', List(String))])

In [69]:
df.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,license_type,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""0""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""7178""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,null,null,53.387852,null,null,null,null,null,null,null,0.657147,null,null,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""",""" Helicobacter pylori Screeni…","""""","""[]""","""2024-10-08""","""["" Artificial intelligence"",""S…",0.0,null,"""AACE Endocrinology and Diabete…","""""","""""","""""","""""","""[""*Brain Trauma Foundation, Pa…",0.0,"""0""","""0""","""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,null,null,31.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,43.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,59.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-5-12""","""[]""",1194.0,null,"""npj Metabolic Health and Disea…","""ecancer Global Foundation""","""©Zongqi Xia, Prerna Chikersal,…","""2026""","""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,"""9""","""9""","""[]""","""[]""",1.0,"""[]""",1.0


In [70]:
# creating a summary dataframe of common metrics
summary_df = pl.DataFrame(
    {
        "Metric": [
            "Total Articles",
            "Unique Authors",
            "Unique Keywords",
            "Avg Authors per Paper",
            "Avg Keywords per Paper",
        ],
        "Value": [
            df_base.select(pl.col("pmid").n_unique()).item(),
            df_authors.select(pl.col("author_full_name").n_unique()).item(),
            df_keywords.select(pl.col("keyword").n_unique()).item(),
            df_authors.group_by("pmid")
            .agg(pl.col("author_full_name").n_unique().alias("n_authors"))
            .select(pl.col("n_authors").mean().round(2))
            .item(),
            df_keywords.group_by("pmid")
            .agg(pl.col("keyword").n_unique().alias("n_keywords"))
            .select(pl.col("n_keywords").mean().round(2))
            .item(),
        ],
    },
    strict=False,
)

summary_df

Metric,Value
str,f64
"""Total Articles""",7178.0
"""Unique Authors""",72964.0
"""Unique Keywords""",16177.0
"""Avg Authors per Paper""",11.47
"""Avg Keywords per Paper""",4.99


#### Building Visuals

In [71]:
data_avail = df_base.group_by("data_available").agg(pl.len().alias("count"))

data_avail

data_available,count
bool,u32
false,4338
true,2840


In [72]:
pub_year_counts = (
    df_base.with_columns(pl.col("pub_date").dt.year().alias("pub_year"))
    .group_by("pub_year")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_year")
)

pub_year_counts

pub_year,num_articles
i32,u32
null,22
2024,91
2025,7058
2026,7


In [73]:
pub_month_counts = (
    df_base
    .with_columns(
        pl.col("pub_date")
          .dt.truncate("1mo")
          .alias("pub_month")
    )
    .group_by("pub_month")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_month")
)

pub_month_counts

pub_month,num_articles
date,u32
null,22
2024-03-01,1
2024-04-01,1
2024-05-01,2
2024-07-01,7
…,…
2025-11-01,643
2025-12-01,566
2026-01-01,4


In [74]:
top_keywords = (
    df_keywords
    .filter(pl.col("keyword").is_not_null())
    .group_by("keyword")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_keywords

keyword,count
str,u32
"""machine learning""",527
"""artificial intelligence""",425
"""epidemiology""",201
"""mortality""",181
"""electronic health records""",181
…,…
"""sepsis""",113
"""breast cancer""",107
"""digital health""",106


In [75]:
top_authors = (
    df_authors
    .filter(pl.col("author_full_name").is_not_null())
    .group_by("author_full_name")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_authors

author_full_name,count
str,u32
""" """,284
"""Kiyohide Fushimi""",45
"""Hideo Yasunaga""",24
"""Hiroki Matsui""",21
"""Jiang Bian""",21
…,…
"""Wei Wang""",13
"""Xin Wang""",13
"""Phichayut Phinyo""",13


#### Visualizing

In [76]:
(
    alt.Chart(data_avail)
    .mark_bar()
    .encode(
        x=alt.X("data_available:O", title=""),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Data Availability")
)

alt.Chart(...)

In [77]:
(
    alt.Chart(pub_year_counts)
    .mark_bar()
    .encode(
        x=alt.X("pub_year:O", title="Publication Year"),
        y=alt.Y("num_articles:Q", title="Number of Articles"),
    )
    .properties(title="Publications per Year")
)

alt.Chart(...)

In [78]:
(
    alt.Chart(pub_month_counts)
    .mark_line(point=True)
    .encode(
        x=alt.X("pub_month:T", title="Publication Date"),
        y=alt.Y("num_articles:Q", title="Number of Articles")
    )
    .properties(title="Publications Over Time")
)

alt.Chart(...)

In [79]:
(
    alt.Chart(top_keywords)
    .mark_bar()
    .encode(
        x=alt.X(
            "keyword:O",
            title="Keyword",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Count"),
    )
    .properties(title="Top 15 Keywords")
)

alt.Chart(...)

In [80]:
(
    alt.Chart(top_authors)
    .mark_bar()
    .encode(
        x=alt.X(
            "author_full_name:O",
            title="Author",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Top 15 Authors")
)

alt.Chart(...)